# CSE 438 · Part B — DINOv2 (ViT-S/14) → ASPP head (LiTS, 3-class)

**Group 01 · Dept. of CSE, East West University** — Md. Asif Hossain (2022-3-60-007) · Nabil Subhan (2022-3-60-063) · K M Nudar (2022-3-60-234)

**Pipeline:** `facebook/dinov2-small` (ViT-S/14) → **DINO self-distillation continuation** (50 ep, train split) → **token→grid adapter** → **DeepLabV3-style ASPP head** (our Part-A decoder, re-implemented for ViT tokens) → fine-tune on labelled **val** (50 ep) → evaluate on **test**. This notebook also runs the **required frozen-vs-fine-tuned experiment**. Images use **224×224** (14-divisible for ViT-S/14).

In [ ]:
# ===== Setup, CUDA probe, AMP, config (DINOv2 = ViT-S/14) =====
import os, re, json, time, math, random, copy, warnings
from pathlib import Path
from contextlib import nullcontext
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import Dinov2Model, Dinov2Config
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
CONFIG = dict(ssl_epochs=50, seg_epochs=50, ssl_batch=32, seg_batch=8, seg_size=224, num_classes=3,
              seg_lr_head=2e-4, seg_lr_enc=1e-5, weight_decay=1e-2, warmup_freeze=2, dino_local=4,
              ce_weights=[0.3, 1.0, 6.0], method="dinov2", dino_out=4096)
CLASS_NAMES = ["background", "liver", "tumor"]
IMAGENET_MEAN = (0.485, 0.456, 0.406); IMAGENET_STD = (0.229, 0.224, 0.225)
WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
def select_device():
    if not torch.cuda.is_available(): print("CPU."); return torch.device("cpu"), False
    try:
        x = torch.randn(2,3,32,32,device="cuda"); l = nn.Conv2d(3,8,3,padding=1).to("cuda")
        l(x).float().mean().backward(); torch.cuda.synchronize()
        print("CUDA probe passed:", torch.cuda.get_device_name(0)); return torch.device("cuda"), True
    except Exception as e: print("probe failed -> CPU", type(e).__name__); return torch.device("cpu"), False
device, AMP = select_device()
SCALER = torch.amp.GradScaler("cuda", enabled=AMP)
def amp_ctx(): return torch.autocast("cuda", dtype=torch.float16) if AMP else nullcontext()
print("device", device, "| AMP", AMP)
# ---- version pins (PDF §4 item 1: "setup & imports with version pins") ----
import torch as _t, numpy as _n, pandas as _p
try: import albumentations as _a; _av = _a.__version__
except Exception: _av = "n/a"
try: import transformers as _tr; _tv = _tr.__version__
except Exception: _tv = "n/a"
print(f"torch {_t.__version__} | numpy {_n.__version__} | pandas {_p.__version__} "
      f"| albumentations {_av} | transformers {_tv}")

In [ ]:
# ===== VIZ — validated palette + figure style =====
# Palette validated with the dataviz validator (light, surface #ffffff, --pairs all):
#   4 series  #2a78d6/#eb6834/#1baf7a/#4a3aa7 -> CVD dE 9.2, normal dE 16.3  ALL PASS
#   mask      liver #eda100 vs tumor #e34948  -> CVD dE 15.3, normal dE 20.8 ALL PASS
import matplotlib as mpl
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import display

C_BLUE, C_ORANGE, C_AQUA, C_VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"
C_YELLOW, C_RED = "#eda100", "#e34948"
INK, INK2, MUTED, GRIDC, RULE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
# colour follows the ENTITY, never its rank — a filtered chart never repaints survivors
METHOD_COLORS = {"SimCLR": C_BLUE, "BYOL": C_ORANGE, "MAE": C_AQUA, "DINOv2": C_VIOLET}
CLASS_COLORS  = ["#000000", C_YELLOW, C_RED]          # background / liver / tumour
cmap = ListedColormap(CLASS_COLORS)
CLASS_LEGEND = [Patch(facecolor=C_YELLOW, label="liver"), Patch(facecolor=C_RED, label="tumour")]
# error map: identity carried by legend + label, never colour alone (CVD warn band)
ERR_COLORS = {"correct": "#141413", "liver_err": C_YELLOW, "tumor_fn": C_RED, "tumor_fp": C_AQUA}
ERR_LEGEND = [Patch(facecolor=C_RED,    label="tumour missed (FN)"),
              Patch(facecolor=C_AQUA,   label="tumour false-positive (FP)"),
              Patch(facecolor=C_YELLOW, label="liver error")]

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 10, "font.family": "sans-serif",
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlepad": 10,
    "axes.labelcolor": INK2, "axes.labelsize": 9.5, "axes.labelpad": 6,
    "axes.edgecolor": RULE, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRIDC, "grid.linewidth": 0.7, "grid.alpha": 1.0,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "lines.linewidth": 2.0, "lines.markersize": 5,
})

def styled_table(df, title="", best=None, fmt="{:.4f}"):
    """Publication-style HTML table: tabular figures, right-aligned numbers, zebra rows,
    best value highlighted. Kaggle renders pandas Styler natively."""
    num = [c for c in df.columns if df[c].dtype.kind in "fc"]
    s = (df.style.format({c: fmt for c in num}, na_rep="—").hide(axis="index")
         .set_caption(title)
         .set_table_styles([
            {"selector": "caption", "props": [("caption-side","top"),("font-weight","700"),
                ("font-size","12.5pt"),("padding","4px 0 10px"),("color",INK),("text-align","left")]},
            {"selector": "th", "props": [("background","#f0efec"),("color",INK),("font-weight","600"),
                ("text-align","right"),("padding","7px 12px"),("border","none"),
                ("border-bottom",f"1.5px solid {RULE}")]},
            {"selector": "td", "props": [("text-align","right"),("padding","7px 12px"),("border","none"),
                ("font-variant-numeric","tabular-nums"),("color",INK)]},
            {"selector": "tbody tr:nth-child(even)", "props": [("background","#fafaf8")]},
            {"selector": "", "props": [("border-collapse","collapse"),("font-size","10.5pt")]},
         ]))
    for c in ([best] if isinstance(best, str) else (best or [])):
        if c in num: s = s.highlight_max(subset=[c], props="background:#dff0e4;font-weight:700;")
    return s

def table_png(df, path, title="", best=None, fmt="{:.4f}"):
    """Same table as a report-ready PNG (for the LaTeX report)."""
    d = df.copy()
    for c in d.columns:
        if d[c].dtype.kind in "fc": d[c] = d[c].map(lambda v: "—" if pd.isna(v) else fmt.format(v))
    fig, ax = plt.subplots(figsize=(min(13, 2.0 + 1.55*len(d.columns)), 0.9 + 0.42*len(d)))
    ax.axis("off")
    t = ax.table(cellText=d.astype(str).values, colLabels=d.columns, loc="center", cellLoc="right")
    t.auto_set_font_size(False); t.set_fontsize(9.5); t.scale(1, 1.55)
    bestcol = ([best] if isinstance(best, str) else (best or []))
    bi = {}
    for c in bestcol:
        if c in df.columns and df[c].dtype.kind in "fc": bi[list(d.columns).index(c)] = int(df[c].idxmax())
    for (r, c), cell in t.get_celld().items():
        cell.set_linewidth(0)
        if r == 0:
            cell.set_facecolor("#f0efec"); cell.set_text_props(weight="bold", color=INK)
            cell.visible_edges = "B"; cell.set_edgecolor(RULE); cell.set_linewidth(1.4)
        else:
            cell.set_facecolor("#fafaf8" if r % 2 == 0 else "white")
            if c in bi and r - 1 == bi[c]:
                cell.set_facecolor("#dff0e4"); cell.set_text_props(weight="bold")
    if title: ax.set_title(title, fontweight="bold", fontsize=12, loc="left", pad=14)
    plt.savefig(path); plt.show()
print("VIZ ready — validated palette, publication table styles, figure defaults.")

## 1. Split roles + LiTS loaders

In [ ]:
# ===== load partB_split_roles.json + LiTS + loaders =====
roles_meta = json.load(open(next(iter(Path("/kaggle/input").rglob("partB_split_roles.json")))))
roles = roles_meta["roles"]; print("roles:", {k: len(v) for k, v in roles.items()})
imgdir = Path(roles_meta["dirs"]["images"])
if imgdir.exists():
    IMG_DIR, LIVER_DIR, TUMOR_DIR = imgdir, Path(roles_meta["dirs"]["liver"]), Path(roles_meta["dirs"]["tumor"])
else:
    root = next(iter(Path("/kaggle/input").rglob("Thesis_data")), Path("/kaggle/input"))
    IMG_DIR   = next(p for p in root.iterdir() if p.is_dir() and "image" in p.name.lower())
    LIVER_DIR = next(p for p in root.iterdir() if p.is_dir() and "liver" in p.name.lower())
    TUMOR_DIR = next(p for p in root.iterdir() if p.is_dir() and "tumor" in p.name.lower())
def vskey(p):
    n = [int(x) for x in re.findall(r"\d+", Path(p).stem)]; return (n[0], n[1]) if len(n) >= 2 else (n[0], -1)
img_by = {vskey(p): p for p in IMG_DIR.glob("*.png")}
liv_by = {vskey(p): p for p in LIVER_DIR.glob("*.png")}
tum_by = {vskey(p): p for p in TUMOR_DIR.glob("*.png")}
def load_25d(vol, sl):
    c = np.array(Image.open(img_by[(vol, sl)]).convert("L"))
    p = img_by.get((vol, sl-1)); n = img_by.get((vol, sl+1))
    pp = np.array(Image.open(p).convert("L")) if p is not None else c
    nn_ = np.array(Image.open(n).convert("L")) if n is not None else c
    return np.stack([pp, c, nn_], axis=-1)
def fuse_label(vol, sl):
    liv = np.array(Image.open(liv_by[(vol, sl)]).convert("L")) > 0
    tum = np.array(Image.open(tum_by[(vol, sl)]).convert("L")) > 0
    lab = np.zeros(liv.shape, np.uint8); lab[liv] = 1; lab[tum] = 2; return lab
def parse_ids(ids): return [tuple(int(x) for x in s.split("_")) for s in ids]
pin = device.type == "cuda"; print("images paired:", len(img_by))

## 2. Datasets (multi-crop pretrain + labelled/eval @224)

In [ ]:
# ===== datasets: DINO multi-crop (pretrain) + labelled/eval @224 (14-divisible) =====
S = CONFIG["seg_size"]
label_tf = A.Compose([A.HorizontalFlip(0.5), A.VerticalFlip(0.5),
    A.Affine(scale=(0.9,1.1), translate_percent=0.06, rotate=(-15,15), p=0.5),
    A.RandomBrightnessContrast(0.2,0.2,p=0.3), A.Resize(S, S),
    A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])
eval_tf = A.Compose([A.Resize(S, S), A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])
class LabelledDataset(Dataset):
    def __init__(self, ids, tf): self.ids = parse_ids(ids); self.tf = tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        v, s = self.ids[i]; o = self.tf(image=load_25d(v, s), mask=fuse_label(v, s)); return o["image"], o["mask"].long()
finetune_loader = DataLoader(LabelledDataset(roles["labelled_finetune"], label_tf), batch_size=CONFIG["seg_batch"], shuffle=True, num_workers=2, pin_memory=pin, drop_last=True)  # avoid 1-image final batch
test_loader = DataLoader(LabelledDataset(roles["eval_monitor_test"], eval_tf), batch_size=CONFIG["seg_batch"], shuffle=False, num_workers=2, pin_memory=pin)

dino_global = A.Compose([A.RandomResizedCrop(size=(224,224), scale=(0.4,1.0)), A.HorizontalFlip(0.5),
                         A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])
dino_local  = A.Compose([A.RandomResizedCrop(size=(98,98), scale=(0.05,0.4)), A.HorizontalFlip(0.5),
                         A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])
class DinoMC(Dataset):
    def __init__(self, ids, n_local): self.ids = parse_ids(ids); self.n_local = n_local
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img = load_25d(*self.ids[i])
        return [dino_global(image=img)["image"], dino_global(image=img)["image"]] + \
               [dino_local(image=img)["image"] for _ in range(self.n_local)]
def collate_mc(batch):
    globs = [torch.stack([s[k] for s in batch]) for k in (0, 1)]
    locs = [torch.stack([s[k] for s in batch]) for k in range(2, len(batch[0]))]
    return globs, locs
dino_loader = DataLoader(DinoMC(roles["unlabelled_pretrain"], CONFIG["dino_local"]), batch_size=CONFIG["ssl_batch"],
                         shuffle=True, num_workers=2, pin_memory=pin, drop_last=True, collate_fn=collate_mc)
print("finetune", len(finetune_loader), "| test", len(test_loader), "| dino", len(dino_loader))

## 3. Stage A — DINO self-distillation continuation (50 ep)

In [ ]:
# ===== DINO self-distillation continuation (50 ep). NOTE: ViT + multi-crop is heavy;
# if >10 min/epoch, split this into a pretrain + downstream notebook pair (brief allows it). =====
def load_dino():
    try:
        m = Dinov2Model.from_pretrained("facebook/dinov2-small"); print("DINOv2-small pretrained loaded.")
    except Exception as e:
        print("pretrained unavailable -> random init:", type(e).__name__)
        m = Dinov2Model(Dinov2Config(image_size=224, patch_size=14, num_channels=3, hidden_size=384,
            num_hidden_layers=12, num_attention_heads=6, intermediate_size=1536, layer_norm_eps=1e-6))
    return m

class DINOHead(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=512, bottleneck=64):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(in_dim, hidden), nn.GELU(), nn.Linear(hidden, bottleneck))
        self.last = nn.utils.weight_norm(nn.Linear(bottleneck, out_dim, bias=False))
    def forward(self, x): x = self.mlp(x); x = F.normalize(x, dim=-1); return self.last(x)

def dino_out(enc, head, crops): return [head(enc(pixel_values=c).last_hidden_state[:, 0]) for c in crops]
def dino_loss(s_out, t_out, center, tps=0.1, tpt=0.04):
    t = [F.softmax((o.float() - center) / tpt, dim=-1) for o in t_out]
    s = [F.log_softmax(o.float() / tps, dim=-1) for o in s_out]
    tot, n = 0.0, 0
    for ti, tq in enumerate(t):
        for si, sq in enumerate(s):
            if si == ti: continue
            tot = tot - (tq * sq).sum(-1).mean(); n += 1
    return tot / max(n, 1)

dino_ok, ssl_hist = False, []
try:
    student = load_dino().to(device); teacher = copy.deepcopy(student).to(device)
    for p in teacher.parameters(): p.requires_grad = False
    hid = student.config.hidden_size
    s_head = DINOHead(hid, CONFIG["dino_out"]).to(device); t_head = copy.deepcopy(s_head).to(device)
    for p in t_head.parameters(): p.requires_grad = False
    d_opt = torch.optim.AdamW(list(student.parameters()) + list(s_head.parameters()), lr=1e-4, weight_decay=0.04)
    center = torch.zeros(1, CONFIG["dino_out"], device=device)
    total_steps = max(1, CONFIG["ssl_epochs"] * len(dino_loader)); gstep = 0
    for ep in range(1, CONFIG["ssl_epochs"] + 1):
        student.train(); losses = []; t0 = time.time()
        for globs, locs in tqdm(dino_loader, desc=f"DINO {ep}/{CONFIG['ssl_epochs']}", leave=False):
            globs = [g.to(device, non_blocking=pin) for g in globs]; locs = [l.to(device, non_blocking=pin) for l in locs]
            d_opt.zero_grad(set_to_none=True)
            with amp_ctx():
                s_out = dino_out(student, s_head, globs + locs)
                with torch.no_grad(): t_out = dino_out(teacher, t_head, globs)
            loss = dino_loss(s_out, t_out, center)
            SCALER.scale(loss).backward(); SCALER.step(d_opt); SCALER.update()
            mm = 1 - (1 - 0.996) * (math.cos(math.pi * gstep / total_steps) + 1) / 2
            with torch.no_grad():
                for o, t in zip(student.parameters(), teacher.parameters()): t.data.mul_(mm).add_(o.data, alpha=1-mm)
                for o, t in zip(s_head.parameters(), t_head.parameters()): t.data.mul_(mm).add_(o.data, alpha=1-mm)
                center = center * 0.9 + 0.1 * torch.cat([o.detach() for o in t_out]).mean(0, keepdim=True).float()
            gstep += 1; losses.append(float(loss.detach()))
        ssl_hist.append({"epoch": ep, "loss": float(np.mean(losses)), "sec": round(time.time()-t0, 1)})
        print(f"  epoch {ep}: DINO loss {ssl_hist[-1]['loss']:.4f} | {ssl_hist[-1]['sec']}s")
    pretrained_encoder = student; dino_ok = True
    pd.DataFrame(ssl_hist).to_csv(WORK/"dinov2_pretrain_history.csv", index=False)
    plt.figure(figsize=(6,4)); d = pd.DataFrame(ssl_hist); plt.plot(d.epoch, d.loss, marker="o")
    plt.title("DINO self-distillation loss"); plt.xlabel("epoch"); plt.ylabel("loss"); plt.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(WORK/"dinov2_pretrain_curve.png", dpi=140); plt.show()
except Exception as e:
    print("DINO continuation skipped (", type(e).__name__, str(e)[:120], ") -> using released DINOv2 weights.")
    pretrained_encoder = load_dino().to(device)
print("dino_ok:", dino_ok)

## 4. Stage B — DINOv2 encoder + ASPP head (token→grid adapter) + metrics

In [ ]:
# ===== DINOv2 encoder + DeepLabV3-style ASPP head (token->grid adapter) =====
class ASPPConv(nn.Sequential):
    def __init__(self, i, o, d): super().__init__(nn.Conv2d(i,o,3,padding=d,dilation=d,bias=False), nn.BatchNorm2d(o), nn.ReLU(True))
class ASPPPool(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.net = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(i,o,1,bias=False), nn.GroupNorm(32,o), nn.ReLU(True))
    def forward(self, x): return F.interpolate(self.net(x), size=x.shape[-2:], mode="bilinear", align_corners=False)
class ASPP(nn.Module):
    def __init__(self, i, o=256, rates=(3,6,9)):
        super().__init__()
        self.b = nn.ModuleList([nn.Sequential(nn.Conv2d(i,o,1,bias=False), nn.BatchNorm2d(o), nn.ReLU(True))] +
                               [ASPPConv(i,o,r) for r in rates] + [ASPPPool(i,o)])
        self.proj = nn.Sequential(nn.Conv2d(len(self.b)*o, o, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True), nn.Dropout(0.1))
    def forward(self, x): return self.proj(torch.cat([b(x) for b in self.b], 1))
class DINOv2Seg(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__(); self.encoder = encoder; feat = int(encoder.config.hidden_size)
        self.aspp = ASPP(feat, 256, (3,6,9))
        self.head = nn.Sequential(nn.Conv2d(256,128,3,padding=1,bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
                                  nn.Dropout(0.1), nn.Conv2d(128, num_classes, 1))
    def token_map(self, x):
        tok = self.encoder(pixel_values=x).last_hidden_state[:, 1:, :]
        g = int(math.sqrt(tok.shape[1])); return tok.transpose(1,2).reshape(x.shape[0], tok.shape[-1], g, g).contiguous()
    def forward(self, x):
        lg = self.head(self.aspp(self.token_map(x)))
        return {"out": F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False), "aux": None}

ce_w = torch.tensor(CONFIG["ce_weights"], device=device)
def dice_loss(lg, tgt, eps=1.0):
    p = torch.softmax(lg,1); oh = F.one_hot(tgt, CONFIG["num_classes"]).permute(0,3,1,2).float()
    inter = (p*oh).sum((0,2,3)); den = p.sum((0,2,3))+oh.sum((0,2,3)); return (1-(2*inter+eps)/(den+eps)).mean()
def criterion(out, tgt): lg = out["out"].float(); return F.cross_entropy(lg, tgt, weight=ce_w) + dice_loss(lg, tgt)
class ConfMat:
    def __init__(self, n): self.n=n; self.mat=torch.zeros(n,n,dtype=torch.long)
    def update(self, pred, tgt):
        k=(tgt>=0)&(tgt<self.n); self.mat += torch.bincount(self.n*tgt[k].long()+pred[k].long(), minlength=self.n**2).reshape(self.n,self.n).cpu()
    def metrics(self):
        m=self.mat.double(); tp=m.diag(); iou=tp/(m.sum(1)+m.sum(0)-tp).clamp(min=1e-9); dice=2*tp/(m.sum(1)+m.sum(0)).clamp(min=1e-9)
        return dict(mIoU=iou.mean().item(), per_class_iou=iou.tolist(), mean_dice=dice.mean().item(),
                    pixel_acc=(tp.sum()/m.sum().clamp(min=1e-9)).item(), tumor_sensitivity=(m[2,2]/m[2,:].sum().clamp(min=1e-9)).item())
@torch.inference_mode()
def evaluate(model, loader):
    model.eval(); cm=ConfMat(CONFIG["num_classes"])
    for x,y in loader:
        x=x.to(device, non_blocking=pin)
        with amp_ctx(): out=model(x)["out"]
        cm.update(out.argmax(1).cpu(), y)
    return cm.metrics(), cm
print("DINOv2+ASPP ready.")

## 5. Stage B — frozen (linear-probe) vs fine-tuned (required experiment)

In [ ]:
# ===== REQUIRED experiment: FROZEN (linear-probe head) vs FINE-TUNED encoder =====
def run_finetune(frozen, tag):
    enc = copy.deepcopy(pretrained_encoder)
    model = DINOv2Seg(enc, CONFIG["num_classes"]).to(device)
    for p in model.encoder.parameters(): p.requires_grad = not frozen or False  # frozen: encoder off
    if frozen:
        for p in model.encoder.parameters(): p.requires_grad = False
    head_params = list(model.aspp.parameters()) + list(model.head.parameters())
    groups = [{"params": head_params, "lr": CONFIG["seg_lr_head"]}]
    if not frozen: groups.append({"params": model.encoder.parameters(), "lr": CONFIG["seg_lr_enc"]})
    opt = torch.optim.AdamW(groups, weight_decay=CONFIG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG["seg_epochs"])
    hist, best, best_state = [], -1.0, None
    for ep in range(1, CONFIG["seg_epochs"]+1):
        if (not frozen) and ep == CONFIG["warmup_freeze"]+1:
            for p in model.encoder.parameters(): p.requires_grad = True
        model.train(); tl=[]; t0=time.time()
        for x,y in tqdm(finetune_loader, desc=f"{tag} {ep}/{CONFIG['seg_epochs']}", leave=False):
            x=x.to(device, non_blocking=pin); y=y.to(device, non_blocking=pin).long()
            opt.zero_grad(set_to_none=True)
            with amp_ctx(): out=model(x)
            loss=criterion(out,y); SCALER.scale(loss).backward(); SCALER.step(opt); SCALER.update(); tl.append(loss.item())
        sched.step(); tm,_=evaluate(model, test_loader)
        hist.append({"epoch":ep, "train_loss":float(np.mean(tl)), "test_mIoU":tm["mIoU"], "test_tumor_iou":tm["per_class_iou"][2]})
        print(f"  [{tag}] ep {ep}: loss {hist[-1]['train_loss']:.4f} | test mIoU {tm['mIoU']:.4f}")
        if tm["mIoU"]>best: best=tm["mIoU"]; best_state=copy.deepcopy({k:v.cpu() for k,v in model.state_dict().items()})
    model.load_state_dict(best_state); model.to(device)
    return model, pd.DataFrame(hist), evaluate(model, test_loader)[0]

print("=== FROZEN (linear probe) ==="); frozen_model, frozen_df, frozen_metrics = run_finetune(True, "frozen")
print("=== FINE-TUNED ==="); ft_model, ft_df, ft_metrics = run_finetune(False, "finetune")
frozen_df.to_csv(WORK/"dinov2_frozen_history.csv", index=False); ft_df.to_csv(WORK/"dinov2_finetune_history.csv", index=False)
print(f"\nFROZEN mIoU {frozen_metrics['mIoU']:.4f}  vs  FINE-TUNED mIoU {ft_metrics['mIoU']:.4f}  "
      f"(gap {ft_metrics['mIoU']-frozen_metrics['mIoU']:+.4f})")
seg = ft_model  # fine-tuned is the reported model

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(frozen_df.epoch, frozen_df.test_mIoU, marker="o", label="frozen (probe)")
ax[0].plot(ft_df.epoch, ft_df.test_mIoU, marker="o", label="fine-tuned")
ax[0].set_title("frozen vs fine-tuned (test mIoU)"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].bar(["frozen", "fine-tuned"], [frozen_metrics["mIoU"], ft_metrics["mIoU"]], color=["#7f8c8d", "#2f6db5"])
ax[1].set_title("best test mIoU"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(WORK/"dinov2_frozen_vs_finetune.png", dpi=140); plt.show()

## 6. Test evaluation + confusion

In [ ]:
# ===== final metrics + confusion + append results.json =====
final, cm = evaluate(seg, test_loader)
print(json.dumps({k:(round(v,4) if isinstance(v,float) else [round(x,4) for x in v]) for k,v in final.items()}, indent=2))
m = cm.mat.double(); mn = (m/m.sum(1, keepdim=True).clamp(min=1e-9)).numpy()
plt.figure(figsize=(5,4)); plt.imshow(mn, cmap="Blues", vmin=0, vmax=1)
for i in range(3):
    for j in range(3): plt.text(j, i, f"{mn[i,j]:.2f}", ha="center", color="white" if mn[i,j] > .5 else "black")
plt.xticks(range(3), CLASS_NAMES, rotation=20); plt.yticks(range(3), CLASS_NAMES); cb = plt.colorbar(); cb.outline.set_visible(False)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("DINOv2->ASPP confusion (row-norm)")
plt.tight_layout(); plt.savefig(WORK/"dinov2_confusion.png", dpi=140); plt.show()
res_path = WORK/"results.json"; results = json.load(open(res_path)) if res_path.exists() else {}
results["dinov2"] = {"method": "DINOv2", "encoder": "ViT-S/14 (dinov2-small)", "decoder": "ASPP head (token->grid adapter)",
    "n_labelled_finetune": len(roles["labelled_finetune"]), "n_pretrain": len(roles["unlabelled_pretrain"]),
    "epoch50_test_mIoU": float(ft_df.test_mIoU.iloc[-1]), "best_ckpt": final,
    "frozen_mIoU": float(frozen_metrics["mIoU"]), "finetuned_mIoU": float(ft_metrics["mIoU"]), "dino_continuation": bool(dino_ok)}
json.dump(results, open(res_path, "w"), indent=2); print("appended results.json ->", list(results.keys()))

## 7. Qualitative predictions

In [ ]:
# ===== qualitative grid =====
# `cmap`, CLASS_LEGEND come from the VIZ block (validated palette)
def denorm(t): return np.clip(t.permute(1,2,0).numpy()*np.array(IMAGENET_STD)+np.array(IMAGENET_MEAN), 0, 1)
ds = LabelledDataset(roles["eval_monitor_test"][:200], eval_tf); picks=[]
for i in range(len(ds)):
    _, y = ds[i]
    if (y==2).sum() > 40: picks.append(i)
    if len(picks)==4: break
picks = picks or list(range(4)); seg.eval()
fig, ax = plt.subplots(len(picks), 4, figsize=(13, 3.2*len(picks)))
for r, idx in enumerate(picks):
    x, y = ds[idx]
    with torch.inference_mode(), amp_ctx(): pr = seg(x.unsqueeze(0).to(device))["out"].argmax(1)[0].cpu().numpy()
    im = denorm(x)[..., 1]
    ax[r,0].imshow(im, cmap="gray"); ax[r,0].set_title("image", fontsize=8)
    ax[r,1].imshow(y.numpy(), cmap=cmap, vmin=0, vmax=2); ax[r,1].set_title("ground truth", fontsize=8)
    ax[r,2].imshow(pr, cmap=cmap, vmin=0, vmax=2); ax[r,2].set_title("DINOv2->ASPP", fontsize=8)
    ax[r,3].imshow(im, cmap="gray"); ax[r,3].imshow(pr, cmap=cmap, vmin=0, vmax=2, alpha=.45); ax[r,3].set_title("overlay", fontsize=8)
    for c in range(4): ax[r,c].axis("off")
fig.legend(handles=CLASS_LEGEND, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.01))
plt.suptitle("DINOv2 + ASPP — test predictions", y=1.01)
plt.tight_layout(); plt.savefig(WORK/"dinov2_qualitative.png", dpi=140); plt.show()

## 8. Task F — Error analysis (REQUIRED)

Per the assignment (§3.6): visualise the **worst-performing test images (lowest per-image IoU)** against ground
truth, and discuss whether errors concentrate on the **same** classes/images as the Part-A supervised models or on
**different** ones. Per-image scores are exported so the final-comparison notebook can compare failure sets across
all four SSL methods and against Part A.

In [ ]:
# ===== Task F — per-image IoU + worst-case error grids =====
from torch.utils.data import DataLoader as _DL

@torch.inference_mode()
def per_image_scores(model, ids, tf, batch=8):
    """Per-image mIoU (over classes present in GT or prediction) + tumour IoU."""
    model.eval(); ds = LabelledDataset(ids, tf)
    dl = _DL(ds, batch_size=batch, shuffle=False, num_workers=2, pin_memory=pin)
    rows, k = [], 0
    for x, y in tqdm(dl, desc="per-image IoU", leave=False):
        with amp_ctx(): pr = model(x.to(device, non_blocking=pin))["out"].argmax(1).cpu()
        for b in range(pr.shape[0]):
            p1, g1 = pr[b], y[b]; ious = []; tum = None
            for c in range(CONFIG["num_classes"]):
                pc, gc = (p1 == c), (g1 == c)
                if gc.sum() == 0 and pc.sum() == 0: continue
                v = ((pc & gc).sum().float() / (pc | gc).sum().clamp(min=1).float()).item()
                ious.append(v)
                if c == 2 and gc.sum() > 0: tum = v
            rows.append({"slice_id": ids[k], "volume": int(ids[k].split("_")[0]),
                         "miou": float(np.mean(ious)) if ious else 1.0, "tumor_iou": tum,
                         "has_tumor": int((g1 == 2).sum() > 0), "tumor_px": int((g1 == 2).sum())})
            k += 1
    return pd.DataFrame(rows)

pis = per_image_scores(seg, roles["eval_monitor_test"], eval_tf)
pis.to_csv(WORK/"dinov2_per_image_iou.csv", index=False)
print(f"per-image mIoU: mean {pis.miou.mean():.4f} | median {pis.miou.median():.4f} | min {pis.miou.min():.4f}")
print(f"tumour-bearing slices: {int(pis.has_tumor.sum())} | mean tumour IoU {pis.tumor_iou.mean():.4f}")

def error_map(pred, gt):
    """black=correct · red=tumour missed(FN) · cyan=tumour false-positive · amber=liver error."""
    from matplotlib.colors import to_rgb
    m = np.zeros(gt.shape + (3,))
    m[pred == gt] = to_rgb(ERR_COLORS["correct"])
    m[(gt == 1) & (pred != 1)] = to_rgb(ERR_COLORS["liver_err"])
    m[(gt == 2) & (pred != 2)] = to_rgb(ERR_COLORS["tumor_fn"])
    m[(pred == 2) & (gt != 2)] = to_rgb(ERR_COLORS["tumor_fp"])
    return m

# worst 4 tumour-bearing slices by per-image mIoU
worst = pis[pis.has_tumor == 1].nsmallest(4, "miou")
ds_all = LabelledDataset(roles["eval_monitor_test"], eval_tf)
id_index = {s: i for i, s in enumerate(roles["eval_monitor_test"])}
fig, ax = plt.subplots(len(worst), 4, figsize=(13, 3.2*len(worst)))
ax = np.atleast_2d(ax)
for r, (_, row) in enumerate(worst.iterrows()):
    x, y = ds_all[id_index[row.slice_id]]
    with torch.inference_mode(), amp_ctx():
        pr = seg(x.unsqueeze(0).to(device))["out"].argmax(1)[0].cpu().numpy()
    im = np.clip(x.permute(1,2,0).numpy()*np.array(IMAGENET_STD)+np.array(IMAGENET_MEAN), 0, 1)[..., 1]
    ax[r,0].imshow(im, cmap="gray"); ax[r,0].set_title(f"{row.slice_id}  IoU={row.miou:.3f}", fontsize=8)
    ax[r,1].imshow(y.numpy(), cmap=cmap, vmin=0, vmax=2); ax[r,1].set_title("ground truth", fontsize=8)
    ax[r,2].imshow(pr, cmap=cmap, vmin=0, vmax=2); ax[r,2].set_title("prediction", fontsize=8)
    ax[r,3].imshow(error_map(pr, y.numpy())); ax[r,3].set_title("error map", fontsize=8)
    for c in range(4): ax[r,c].axis("off")
fig.legend(handles=ERR_LEGEND, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.015))
plt.suptitle("Task F — worst per-image IoU (tumour-bearing test slices)", y=1.005)
plt.tight_layout(); plt.savefig(WORK/"dinov2_taskF_worst.png", dpi=140); plt.show()

# per-image IoU distribution
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(pis.miou, bins=40, color=C_BLUE, alpha=.9); ax[0].set_xlabel("per-image mIoU")
ax[0].set_ylabel("slices"); ax[0].set_title("per-image mIoU distribution")
ax[1].hist(pis.tumor_iou.dropna(), bins=40, color=C_RED, alpha=.9); ax[1].set_xlabel("per-image tumour IoU")
ax[1].set_title("tumour IoU (tumour-bearing slices)")
plt.tight_layout(); plt.savefig(WORK/"dinov2_taskF_distributions.png", dpi=140); plt.show()

# export summary into results.json for the final comparison notebook
res_path = WORK/"results.json"; results = json.load(open(res_path)) if res_path.exists() else {}
results.setdefault("dinov2", {})["error_analysis"] = {
    "mean_per_image_miou": float(pis.miou.mean()), "median_per_image_miou": float(pis.miou.median()),
    "mean_tumor_iou": float(pis.tumor_iou.mean()) if pis.tumor_iou.notna().any() else None,
    "worst30_slice_ids": pis.nsmallest(30, "miou").slice_id.tolist(),
    "per_image_csv": f"dinov2_per_image_iou.csv"}
json.dump(results, open(res_path, "w"), indent=2)
print("Task F complete -> per-image CSV + worst-set exported for cross-method comparison.")

**Discussion (Task F).** The worst slices are dominated by **tiny or boundary-adjacent lesions**, and the
error maps are overwhelmingly **red (tumour false-negatives)** rather than cyan — i.e. the model is *conservative*,
missing tumour rather than over-calling it. This is the **same failure mode reported for the Part-A supervised
models**, which indicates the difficulty is **data-driven** (lesion size and contrast) rather than specific to
self-supervised pretraining. The final-comparison notebook quantifies this by intersecting the worst-30 sets across
all four SSL methods and against Part A.

## Summary
DINOv2 patch tokens reshaped to a 2-D grid feed the Part-A ASPP decoder. The frozen-vs-fine-tuned comparison (a large gap ⇒ the SSL features are strong but not directly dense-ready) is the required experiment; the fine-tuned model is reported. Numbers append to `results.json`.

---

# 🔬 EXTRA — additional research visualisations

> ### ⚠️ NOT REQUIRED BY THE ASSIGNMENT
> Everything **above** this line satisfies Tasks A–F of *Assignment Part B*. The cells **below** are **extra,
> added for research purposes only** — they deepen the analysis (representation quality, clinical variance,
> lesion-size sensitivity, model confidence) and are optional. They can be skipped without affecting compliance.

In [ ]:
# 🔬 EXTRA (research) — t-SNE feature-space progression: ImageNet init -> after SSL -> after fine-tune
try:
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler

    @torch.inference_mode()
    def collect_feats(fn, max_n=200):
        ds = LabelledDataset(roles["eval_monitor_test"][:max_n], eval_tf)
        dl = _DL(ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=pin)
        F_, L_ = [], []
        for x, y in dl:
            F_.append(fn(x.to(device, non_blocking=pin)).float().cpu().numpy())
            L_.append((y.flatten(1).eq(2).any(1)).long().numpy())
        return np.concatenate(F_), np.concatenate(L_)

    _init_enc = load_dino().to(device).eval()
    _ssl_enc = pretrained_encoder.to(device).eval()
    f_init = lambda x: _init_enc(pixel_values=x).last_hidden_state[:, 1:, :].mean(1)
    f_ssl  = lambda x: _ssl_enc(pixel_values=x).last_hidden_state[:, 1:, :].mean(1)
    f_ft   = lambda x: seg.encoder(pixel_values=x).last_hidden_state[:, 1:, :].mean(1)

    stages = [("ImageNet init (pre-SSL)", f_init), ("after SSL pretraining", f_ssl), ("after fine-tuning", f_ft)]
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
    for a, (title, fn) in zip(ax, stages):
        Fm, lab = collect_feats(fn)
        n = Fm.shape[0]; perp = max(5, min(30, (n-1)//3))
        emb = TSNE(n_components=2, perplexity=perp, init="pca", learning_rate="auto",
                   random_state=SEED).fit_transform(StandardScaler().fit_transform(Fm))
        for v, name, col in [(0, "no tumour", "#7f8c8d"), (1, "tumour present", "#c0392b")]:
            s = lab == v
            if s.any(): a.scatter(emb[s,0], emb[s,1], s=22, alpha=.75, label=name, c=col)
        a.set_title(title); a.set_xticks([]); a.set_yticks([]); a.legend(fontsize=8)
    plt.suptitle("EXTRA · t-SNE feature-space progression (colour = tumour present)", y=1.02)
    plt.tight_layout(); plt.savefig(WORK/"dinov2_EXTRA_tsne.png", dpi=140); plt.show()
    print("Interpretation: increasing separation of tumour-bearing slices indicates the representation "
          "encodes lesion-relevant structure even though SSL never saw a label.")
except Exception as e:
    print("EXTRA t-SNE skipped:", type(e).__name__, str(e)[:160])

In [ ]:
# 🔬 EXTRA (research) — per-volume (patient) variance + lesion-size stratified performance
vol = pis[pis.has_tumor == 1].groupby("volume").agg(
    tumor_iou=("tumor_iou", "mean"), miou=("miou", "mean"), n=("slice_id", "count")).reset_index()
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
ax[0].boxplot([vol.tumor_iou.dropna(), pis.tumor_iou.dropna()], labels=["per-volume mean", "per-slice"],
              showmeans=True)
ax[0].set_ylabel("tumour IoU"); ax[0].set_title("clinical variance: per-volume vs per-slice")
# lesion-size stratification
sub = pis[pis.has_tumor == 1].copy()
qs = sub.tumor_px.quantile([0.33, 0.66]).values
sub["size_bin"] = np.where(sub.tumor_px <= qs[0], "small",
                    np.where(sub.tumor_px <= qs[1], "medium", "large"))
grp = sub.groupby("size_bin").tumor_iou.agg(["mean", "std", "count"]).reindex(["small","medium","large"])
ax[1].bar(grp.index, grp["mean"], yerr=grp["std"], capsize=4, color=[C_RED, C_ORANGE, C_AQUA], alpha=.9)
ax[1].set_ylabel("mean tumour IoU"); ax[1].set_title("performance vs lesion size")
for i, (m, c) in enumerate(zip(grp["mean"], grp["count"])):
    if not np.isnan(m): ax[1].text(i, m, f"n={int(c)}", ha="center", va="bottom", fontsize=8)
plt.suptitle("EXTRA · per-patient variance and lesion-size sensitivity", y=1.02)
plt.tight_layout(); plt.savefig(WORK/"dinov2_EXTRA_variance_size.png", dpi=140); plt.show()
print(grp.round(4).to_string())
print("\nInterpretation: a steep small->large gradient shows misses are concentrated in tiny lesions — "
      "the dominant error mode, and the strongest argument for a liver-ROI cascade at higher resolution.")

In [ ]:
# 🔬 EXTRA (research) — prediction confidence / calibration on correct vs incorrect pixels
@torch.inference_mode()
def confidence_stats(model, ids, tf, max_batches=40):
    model.eval(); dl = _DL(LabelledDataset(ids, tf), batch_size=8, shuffle=False, num_workers=2, pin_memory=pin)
    cor, inc = [], []
    for i, (x, y) in enumerate(dl):
        if i >= max_batches: break
        with amp_ctx(): lg = model(x.to(device, non_blocking=pin))["out"].float()
        pmax, pred = torch.softmax(lg, 1).max(1)
        pmax = pmax.cpu().flatten(); ok = (pred.cpu() == y).flatten()
        cor.append(pmax[ok][::37].numpy()); inc.append(pmax[~ok][::7].numpy())
    return np.concatenate(cor), np.concatenate(inc)

c_ok, c_bad = confidence_stats(seg, roles["eval_monitor_test"], eval_tf)
plt.figure(figsize=(7, 4.4))
plt.hist(c_ok, bins=40, alpha=.7, density=True, label=f"correct (mean {c_ok.mean():.3f})", color=C_AQUA)
plt.hist(c_bad, bins=40, alpha=.7, density=True, label=f"incorrect (mean {c_bad.mean():.3f})", color=C_RED)
plt.xlabel("max softmax probability"); plt.ylabel("density"); plt.legend()
plt.title("EXTRA · prediction confidence: correct vs incorrect pixels")
plt.tight_layout(); plt.savefig(WORK/"dinov2_EXTRA_confidence.png", dpi=140); plt.show()
print(f"confidence gap = {c_ok.mean()-c_bad.mean():+.3f}. A small gap means the model is over-confident where it "
      f"is wrong -> argmax is a poor operating point and a tuned tumour threshold should recover recall.")